# Era Wallet A-roll proxy builder — resumable
Free Google Colab workflow for the large A-roll file already copied into My Drive. It creates a connector-safe MP4 around 205 MiB **in resumable chunks**. If Colab disconnects, reconnect and run the cells again; completed chunks in Drive are skipped, so you do not lose all progress.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, subprocess, hashlib, math, shlex

SOURCE = '/content/drive/MyDrive/ERA_A_ROLL_AUTH_WORKING.mov'
OUTPUT = '/content/drive/MyDrive/ERA_A_ROLL_PROXY_205M.mp4'
SHA_FILE = '/content/drive/MyDrive/ERA_A_ROLL_PROXY_205M.sha256.txt'
CHUNK_DIR = '/content/drive/MyDrive/ERA_A_ROLL_PROXY_CHUNKS'
TARGET_MIB = 205
CHUNK_SECONDS = 180

assert os.path.exists(SOURCE), f'Missing source: {SOURCE}'
os.makedirs(CHUNK_DIR, exist_ok=True)

def sh(cmd):
    return subprocess.check_output(cmd, text=True).strip()

def valid_media(path):
    if not os.path.exists(path) or os.path.getsize(path) < 100_000:
        return False
    try:
        d = float(sh(['ffprobe','-v','error','-show_entries','format=duration','-of','default=nw=1:nk=1', path]))
        return d > 1
    except Exception:
        return False

duration = float(sh(['ffprobe','-v','error','-show_entries','format=duration','-of','default=nw=1:nk=1', SOURCE]))
target_bytes = TARGET_MIB * 1024 * 1024
audio_kbps = 88
video_kbps = max(300, min(2600, int((target_bytes*8/duration/1000) - audio_kbps - 30)))
chunk_count = math.ceil(duration / CHUNK_SECONDS)
print(f'Duration: {duration:.1f}s | video target: {video_kbps} kbps | chunks: {chunk_count}')

# Ignore an unfinished final file from a previous disconnected run.
if os.path.exists(OUTPUT) and not valid_media(OUTPUT):
    os.remove(OUTPUT)

for i in range(chunk_count):
    start = i * CHUNK_SECONDS
    length = min(CHUNK_SECONDS, duration - start)
    chunk = os.path.join(CHUNK_DIR, f'chunk_{i:03d}.mp4')
    part = chunk + '.part.mp4'
    if valid_media(chunk):
        print(f'[{i+1}/{chunk_count}] already done — skipping {os.path.basename(chunk)}')
        continue
    if os.path.exists(part):
        os.remove(part)
    print(f'[{i+1}/{chunk_count}] encoding {start:.0f}s → {start+length:.0f}s')
    cmd = [
        'ffmpeg','-hide_banner','-y',
        '-ss',f'{start:.3f}','-t',f'{length:.3f}','-i',SOURCE,
        '-map','0:v:0','-map','0:a?',
        '-vf',"scale='if(gte(iw,ih),min(1920,iw),-2)':'if(gte(iw,ih),-2,min(1920,ih))',fps=30",
        '-c:v','libx264','-preset','veryfast',
        '-b:v',f'{video_kbps}k','-maxrate',f'{int(video_kbps*1.2)}k','-bufsize',f'{video_kbps*2}k',
        '-pix_fmt','yuv420p','-c:a','aac','-b:a',f'{audio_kbps}k',
        '-avoid_negative_ts','make_zero','-movflags','+faststart',part
    ]
    subprocess.run(cmd, check=True)
    assert valid_media(part), f'Chunk failed validation: {part}'
    os.replace(part, chunk)
    print(f'[{i+1}/{chunk_count}] saved {os.path.basename(chunk)} ({os.path.getsize(chunk)/1024/1024:.1f} MiB)')

missing = [i for i in range(chunk_count) if not valid_media(os.path.join(CHUNK_DIR, f'chunk_{i:03d}.mp4'))]
assert not missing, f'Missing/invalid chunks: {missing}'

concat_file = '/content/era_a_roll_concat.txt'
with open(concat_file,'w') as f:
    for i in range(chunk_count):
        chunk = os.path.join(CHUNK_DIR, f'chunk_{i:03d}.mp4')
        f.write("file '" + chunk.replace("'", "'\\''") + "'\n")

tmp_output = OUTPUT + '.joining.mp4'
if os.path.exists(tmp_output):
    os.remove(tmp_output)
subprocess.run([
    'ffmpeg','-hide_banner','-y','-f','concat','-safe','0','-i',concat_file,
    '-c','copy','-movflags','+faststart',tmp_output
], check=True)
assert valid_media(tmp_output), 'Joined proxy failed validation.'
size = os.path.getsize(tmp_output)
assert size < 235_000_000, f'Output too large: {size} bytes'
os.replace(tmp_output, OUTPUT)

h = hashlib.sha256()
with open(OUTPUT,'rb') as f:
    for chunk in iter(lambda: f.read(8*1024*1024), b''):
        h.update(chunk)
with open(SHA_FILE,'w') as f:
    f.write(h.hexdigest()+'  '+os.path.basename(OUTPUT)+'\n')

print(f'Created {OUTPUT} ({size/1024/1024:.1f} MiB)')
print('SHA256:', h.hexdigest())


When the last cell prints **Created**, the proxy is in My Drive as `ERA_A_ROLL_PROXY_205M.mp4`.

**If Colab disconnects:** reconnect, run the Drive-mount cell again, then run the processing cell again. Finished chunks are stored in `ERA_A_ROLL_PROXY_CHUNKS` in My Drive and will be skipped automatically.
